# PSDRL Value Network Hyperparameter Search

Tunes the value network (`PSDRL/networks/value.py`) using the **best transition model** found by `optuna_transition_hparam_search.ipynb` (or `run_psdrl_search.py`).

**Input to value network**: `cat([raw_obs (25-dim), GRU_hidden_from_best_tm (gru_dim)])` — loaded from `configs/psdrl_transition_params.json` (`gru_dim`, default 64 → input_dim = 89).

**Training targets**: Monte Carlo returns `G_t = Σ γ^k r_{t+k}` computed from full episodes.

**Two objectives:**
- **Obj 1 — `return_error`**: MSE between value network output and MC return on hold-out episodes.
- **Obj 2 — `env_steps_to_accuracy`**: Minimum transitions to achieve `return_error < 0.05`. Probed at 250, 500; falls back to 1000.

**Search space:** `hidden_dim`, `learning_rate`, `discount`, `training_iterations`

In [ ]:
import sys
sys.path.insert(0, '..')

import random, json, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import NSGAIISampler
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

from bsuite.environments import deep_sea
from PSDRL.networks.value import Network as ValueNetwork
from PSDRL.networks.transition import Network as TransitionNetwork

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

def get_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        torch.zeros(1).cuda(); torch.cuda.synchronize(); return 'cuda'
    except RuntimeError:
        return 'cpu'

DEVICE = get_device()
print(f'Device: {DEVICE}')

## Constants and Search Space

| Parameter | Choices | Description |
|---|---|---|
| `hidden_dim` | [32, 64, 128, 256] | MLP layer width |
| `learning_rate` | [1e-4, 5e-4, 1e-3, 3e-3] | Adam learning rate |
| `discount` | [0.9, 0.95, 0.99] | Discount factor γ for MC returns |
| `training_iterations` | [50, 100, 200] | Gradient steps per trial |

> **Value input = obs (25-dim) + GRU hidden from best transition model (gru_dim)**  
> `gru_dim` is loaded from `configs/psdrl_transition_params.json` (default 64).

In [ ]:
DS_SIZE   = 5
N_ACTIONS = 2
OBS_DIM   = DS_SIZE * DS_SIZE   # 25

# --- Load best transition model config ---
tm_cfg_path = Path('configs/psdrl_transition_params.json')
if tm_cfg_path.exists():
    with open(tm_cfg_path) as f:
        tm_params = json.load(f)
    gru_dim    = tm_params['gru_dim']
    tm_hidden  = tm_params['hidden_dim']
    tm_lr      = tm_params['learning_rate']
    tm_window  = tm_params['window_length']
    tr_prior   = tm_params['transition_prior']
    rw_prior   = tm_params['reward_prior']
    print(f'Loaded TM: gru_dim={gru_dim}, hidden_dim={tm_hidden}, lr={tm_lr}')
else:
    gru_dim = 64; tm_hidden = 64; tm_lr = 1e-3; tm_window = 1
    tr_prior = 1e-2; rw_prior = 0.1
    print('Warning: TM config not found, using defaults')

INPUT_DIM = OBS_DIM + gru_dim   # value network input dimension
print(f'Value network input_dim = {OBS_DIM} + {gru_dim} = {INPUT_DIM}')

HIDDEN_DIM_CHOICES = [32, 64, 128, 256]
LR_CHOICES         = [1e-4, 5e-4, 1e-3, 3e-3]
DISCOUNT_CHOICES   = [0.9, 0.95, 0.99]
ITER_CHOICES       = [50, 100, 200]

TIME_LIMIT        = 1000
TEST_SIZE         = 200
PROBE_SIZES       = [250, 500]
TARGET_RET_ERROR  = 0.05
PATIENCE          = 15
BATCH_SIZE        = 16
N_TRIALS          = 40

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

## Data Collection

1. Collect `TIME_LIMIT` (1000) transitions using random policy on DeepSea 5×5.
2. Build the best transition model from `psdrl_transition_params.json` and run episodes through it to collect GRU hidden states at each step.
3. Value network input at step t: `cat([obs_t, h_{t-1}])` — where `h_{t-1}` is the GRU hidden state *before* processing obs_t.
4. Training target: Monte Carlo return `G_t = Σ γ^k r_{t+k}`.

In [ ]:
def collect_episodes(n_transitions, ds_size=5, seed=42):
    env = deep_sea.DeepSea(size=ds_size, deterministic=True, seed=seed)
    episodes, total = [], 0
    while total < n_transitions:
        obs_list, act_list, rew_list, next_obs_list = [], [], [], []
        ts = env.reset()
        obs = torch.zeros(ds_size * ds_size)
        obs[ts.observation.flatten().argmax()] = 1.0
        while True:
            action = random.randint(0, 1)
            next_ts = env.step(action)
            next_obs = torch.zeros(ds_size * ds_size)
            next_obs[next_ts.observation.flatten().argmax()] = 1.0
            obs_list.append(obs); act_list.append(action)
            rew_list.append(float(next_ts.reward))
            next_obs_list.append(next_obs)
            obs = next_obs; total += 1
            if next_ts.last() or total >= n_transitions: break
        episodes.append({
            'obs':      torch.stack(obs_list),
            'actions':  torch.tensor(act_list),
            'rewards':  torch.tensor(rew_list),
            'next_obs': torch.stack(next_obs_list),
        })
    return episodes


def train_tm(tm, episodes, n_iters=200, window_length=1, lr=1e-3,
             patience=20, device=DEVICE):
    """Quick training of transition model for value input generation."""
    optimizer = torch.optim.Adam(tm.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    best, pat = float('inf'), 0
    for it in range(n_iters):
        idx = random.sample(range(len(episodes)), min(BATCH_SIZE, len(episodes)))
        batch = [episodes[i] for i in idx]
        it_loss, n_steps = 0.0, 0
        for ep in batch:
            h = torch.zeros(1, tm.gru_dim, device=device)
            for t in range(0, len(ep['obs']), window_length):
                optimizer.zero_grad()
                w_loss = torch.tensor(0.0, device=device)
                for tt in range(t, min(t + window_length, len(ep['obs']))):
                    s  = ep['obs'][tt].unsqueeze(0).to(device)
                    a  = ep['actions'][tt].long().item()
                    s1 = ep['next_obs'][tt].to(device)
                    r  = ep['rewards'][tt].to(device)
                    act_oh = F.one_hot(torch.tensor([a], device=device), N_ACTIONS).float()
                    sa = torch.cat([s, act_oh], dim=-1)
                    pred, h_new = tm.forward(sa, h)
                    h = h_new
                    w_loss = w_loss + loss_fn(pred[:, :-1], s1.unsqueeze(0)) + loss_fn(pred[:, -1:], r.view(1,1))
                w_loss.backward(); optimizer.step(); h = h.detach()
                it_loss += w_loss.item(); n_steps += 1
        avg = it_loss / max(1, n_steps)
        if avg < best - 1e-5: best, pat = avg, 0
        else:
            pat += 1
            if pat >= patience: break
    return tm


def collect_value_inputs(episodes, tm, discount, device=DEVICE):
    """Run each episode through the best TM to get GRU hidden states.
    Value input at step t = cat([obs_t, h_{t-1}]).
    """
    tm.eval()
    all_inputs, all_returns = [], []
    with torch.no_grad():
        for ep in episodes:
            T = len(ep['obs'])
            h = torch.zeros(1, tm.gru_dim, device=device)
            h_list = []
            for t in range(T):
                h_list.append(h.squeeze(0).cpu())
                s = ep['obs'][t].unsqueeze(0).to(device)
                a = ep['actions'][t].long().item()
                act_oh = F.one_hot(torch.tensor([a], device=device), N_ACTIONS).float()
                sa = torch.cat([s, act_oh], dim=-1)
                _, h = tm.forward(sa, h)
            rew_np = ep['rewards'].numpy()
            G = np.zeros(T)
            G[-1] = rew_np[-1]
            for t in range(T - 2, -1, -1):
                G[t] = rew_np[t] + discount * G[t + 1]
            for t in range(T):
                inp = torch.cat([ep['obs'][t], h_list[t]], dim=-1)
                all_inputs.append(inp)
                all_returns.append(float(G[t]))
    tm.train()
    return torch.stack(all_inputs), torch.tensor(all_returns, dtype=torch.float32)


print('Collecting episodes...')
all_episodes = collect_episodes(TIME_LIMIT)
n_test = max(1, int(len(all_episodes) * 0.2))
train_episodes = all_episodes[:-n_test]
test_episodes  = all_episodes[-n_test:]
print(f'Train episodes: {len(train_episodes)}, Test episodes: {len(test_episodes)}')

# Build and train best transition model for value input generation
print('Training best transition model for GRU hidden state extraction...')
tm_config = {'gru_dim': gru_dim, 'hidden_dim': tm_hidden, 'learning_rate': tm_lr}
best_tm = TransitionNetwork(OBS_DIM, N_ACTIONS, tm_config, DEVICE)
best_tm = train_tm(best_tm, train_episodes, n_iters=300, window_length=tm_window, lr=tm_lr)
print('Transition model ready.')

## Value Network Training and Evaluation

In [ ]:
def episodes_from_n(episodes, n):
    """Return subset of episodes totalling at most n transitions."""
    subset, total = [], 0
    for ep in episodes:
        if total >= n: break
        subset.append(ep); total += len(ep['obs'])
    return subset


def train_value_net(vnet, inputs, returns, n_iters, lr, patience=PATIENCE):
    optimizer = torch.optim.Adam(vnet.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    N = inputs.shape[0]
    best, pat, e_conv = float('inf'), 0, n_iters
    targets = returns.unsqueeze(1)
    for it in range(n_iters):
        idx = torch.randint(0, N, (BATCH_SIZE,))
        x = inputs[idx].to(DEVICE)
        y = targets[idx].to(DEVICE)
        optimizer.zero_grad()
        pred = vnet(x)
        loss = loss_fn(pred, y)
        loss.backward(); optimizer.step()
        l = loss.item()
        if l < best - 1e-6: best, pat, e_conv = l, 0, it + 1
        else:
            pat += 1
            if pat >= patience: break
    return best, e_conv


def eval_value_net(vnet, inputs, returns):
    vnet.eval()
    with torch.no_grad():
        preds = vnet(inputs.to(DEVICE)).squeeze(1).cpu()
    vnet.train()
    return float(F.mse_loss(preds, returns).item())

## Optuna Objective

In [ ]:
def objective(trial):
    hidden_dim = trial.suggest_categorical('hidden_dim',          HIDDEN_DIM_CHOICES)
    lr         = trial.suggest_categorical('learning_rate',       LR_CHOICES)
    discount   = trial.suggest_categorical('discount',            DISCOUNT_CHOICES)
    n_iters    = trial.suggest_categorical('training_iterations', ITER_CHOICES)

    vnet = ValueNetwork(INPUT_DIM, {'hidden_dim': hidden_dim, 'learning_rate': lr}, DEVICE)

    # Obj 1: full dataset
    tr_inp, tr_ret = collect_value_inputs(train_episodes, best_tm, discount)
    te_inp, te_ret = collect_value_inputs(test_episodes,  best_tm, discount)
    best_loss, e_conv = train_value_net(vnet, tr_inp, tr_ret, n_iters, lr)
    ret_err = eval_value_net(vnet, te_inp, te_ret)
    trial.set_user_attr('epochs_to_converge', e_conv)

    # Obj 2: env_steps_to_accuracy
    env_steps = TIME_LIMIT
    for n in PROBE_SIZES:
        p_eps = episodes_from_n(train_episodes, n)
        p_inp, p_ret = collect_value_inputs(p_eps, best_tm, discount)
        p_te_inp, p_te_ret = collect_value_inputs(test_episodes, best_tm, discount)
        p_vnet = ValueNetwork(INPUT_DIM, {'hidden_dim': hidden_dim, 'learning_rate': lr}, DEVICE)
        train_value_net(p_vnet, p_inp, p_ret, n_iters, lr)
        if eval_value_net(p_vnet, p_te_inp, p_te_ret) < TARGET_RET_ERROR:
            env_steps = n; break

    return ret_err, float(env_steps)

In [ ]:
sampler = NSGAIISampler(seed=42)
study = optuna.create_study(
    directions=['minimize', 'minimize'],
    sampler=sampler,
    study_name='psdrl_value',
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'\nCompleted {len(study.trials)} trials')

## Results Analysis

In [ ]:
def trials_to_df(study):
    rows = []
    for t in study.trials:
        if t.values is None: continue
        row = dict(t.params)
        row['return_error']          = t.values[0]
        row['env_steps_to_accuracy'] = t.values[1]
        row['epochs_to_converge']    = t.user_attrs.get('epochs_to_converge', 200)
        row['pareto']                = t in study.best_trials
        rows.append(row)
    return pd.DataFrame(rows)


df = trials_to_df(study)
pareto_df = df[df['pareto']].sort_values('return_error')

print(f'Total trials: {len(df)}')
print(f'Pareto-front trials: {len(pareto_df)}')
pareto_df[['hidden_dim','learning_rate','discount','training_iterations',
           'return_error','env_steps_to_accuracy']].round(5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(df['env_steps_to_accuracy'], df['mean_return_error'],
           c='lightsteelblue', alpha=0.5, s=30, label='All trials')
ax.scatter(pareto_df['env_steps_to_accuracy'], pareto_df['mean_return_error'],
           c='crimson', s=80, zorder=5, label='Pareto front')
ax.set_xlabel('Env Steps to Accuracy (Obj 2)')
ax.set_ylabel('Mean Return Error (Obj 1)')
ax.set_title('PSDRL Value — Pareto Front')
ax.legend()
ax.grid(True, alpha=0.3)

ax2 = axes[1]
for d in sorted(df['discount'].unique()):
    sub = df[df['discount'] == d]
    ax2.scatter(sub['env_steps_to_accuracy'], sub['mean_return_error'],
                alpha=0.6, s=40, label=f'discount={d}')
ax2.set_xlabel('Env Steps to Accuracy')
ax2.set_ylabel('Mean Return Error')
ax2.set_title('Coloured by discount')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Best Model Evaluation

In [ ]:
if len(pareto_df) > 0:
    pdf = pareto_df.copy()
    for col in ['return_error', 'env_steps_to_accuracy']:
        rng = pdf[col].max() - pdf[col].min()
        pdf[col + '_norm'] = (pdf[col] - pdf[col].min()) / (rng if rng > 0 else 1)
    pdf['score'] = pdf['return_error_norm'] + pdf['env_steps_to_accuracy_norm']
    best_row = pdf.loc[pdf['score'].idxmin()]
else:
    best_row = df.loc[df['return_error'].idxmin()]

best_hd    = int(best_row['hidden_dim'])
best_lr    = float(best_row['learning_rate'])
best_disc  = float(best_row['discount'])
best_iters = int(best_row['training_iterations'])

print('Best config:', best_row[['hidden_dim','learning_rate','discount','training_iterations']].to_dict())
print(f'return_error={best_row["return_error"]:.5f}, env_steps={best_row["env_steps_to_accuracy"]:.0f}')

best_net = ValueNetwork(INPUT_DIM, {'hidden_dim': best_hd, 'learning_rate': best_lr}, DEVICE)
tr_inp, tr_ret = collect_value_inputs(train_episodes, best_tm, best_disc)
te_inp, te_ret = collect_value_inputs(test_episodes,  best_tm, best_disc)
final_loss, final_epochs = train_value_net(best_net, tr_inp, tr_ret, best_iters * 2, best_lr)
final_err = eval_value_net(best_net, te_inp, te_ret)

n_params = sum(p.numel() for p in best_net.parameters())
print(f'Retrained → loss={final_loss:.6f}, test_return_error={final_err:.5f}')
print(f'Value network parameters: {n_params:,}')

In [ ]:
# Per-state value prediction vs MC return
best_net.eval()
with torch.no_grad():
    preds_all = best_net(te_inp.to(DEVICE)).squeeze(1).cpu().numpy()
ret_all = te_ret.numpy()

# Group by state (obs argmax)
obs_part = te_inp[:, :OBS_DIM]   # first 25 dims are the obs
state_idx = obs_part.argmax(dim=-1).numpy()
state_preds  = {s: [] for s in range(OBS_DIM)}
state_rets   = {s: [] for s in range(OBS_DIM)}
for i, s in enumerate(state_idx):
    state_preds[s].append(preds_all[i])
    state_rets[s].append(ret_all[i])

best_net.train()

visited = [s for s in range(OBS_DIM) if state_preds[s]]
pred_means = [float(np.mean(state_preds[s])) for s in visited]
ret_means  = [float(np.mean(state_rets[s]))  for s in visited]

fig, ax = plt.subplots(figsize=(max(10, len(visited) * 0.5), 5))
x = range(len(visited))
ax.bar(x, ret_means,  alpha=0.5, label='MC Return (target)', color='steelblue')
ax.bar(x, pred_means, alpha=0.5, label='Value prediction',   color='orange')
ax.set_xticks(list(x))
ax.set_xticklabels([f'{s}\n({s//DS_SIZE},{s%DS_SIZE})' for s in visited], fontsize=8)
ax.set_xlabel('State (row, col)')
ax.set_ylabel('Return / Value')
ax.set_title('Best Value Net — Per-State Prediction vs MC Return')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## Save Optimal Parameters

In [ ]:
optimal_params = {
    'hidden_dim':            int(best_row['hidden_dim']),
    'learning_rate':         float(best_row['learning_rate']),
    'discount':              float(best_row['discount']),
    'training_iterations':   int(best_row['training_iterations']),
    'return_error':          float(best_row['return_error']),
    'env_steps_to_accuracy': int(best_row['env_steps_to_accuracy']),
    'epochs_to_converge':    int(best_row['epochs_to_converge']),
    'final_train_loss':      float(final_loss),
    'final_test_error':      float(final_err),
    'n_parameters':          n_params,
    'input_dim':             INPUT_DIM,
    'gru_dim':               gru_dim,
    'embed_dim':             OBS_DIM,
    'n_actions':             N_ACTIONS,
    'ds_size':               DS_SIZE,
    'n_trials':              N_TRIALS,
    'data_samples':          TIME_LIMIT,
}

out_path = Path('configs/psdrl_value_params.json')
out_path.parent.mkdir(exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(optimal_params, f, indent=2)
print(f'Saved to {out_path}')
print(json.dumps(optimal_params, indent=2))